### Making structured outputs useful


we will ask the llm to list all of the context used
in this case to display the items to the users 
or could do this for regerences to files or web searches for example 

In [13]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient
from langsmith import get_current_run_tree
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [14]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [15]:
class RAG_GenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens, 
            "total_tokens": response.usage.total_tokens
        }

    return response.data[0].embedding

def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="amazon-items-collection-01",
        query=query_embedding, 
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scored = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocess_description"])
        similarity_scored.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])


    return {
        "retrieved_context_ids":  retrieved_context_ids, 
        "retrieved_context": retrieved_context,
        "similarity_scored": similarity_scored,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
    """

    return prompt

from urllib3 import response

def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ], 
        reasoning={"effort": "none"},
        response_model=RAG_GenerationResponse
    )

    return response

def rag_pipeline(question, topk_k=5):

    retrieved_context = retrieve_data(question, k=topk_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer, 
        "question": question, 
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

output = rag_pipeline("Any usb chargeable devices?")

output 

{'data_object': RAG_GenerationResponse(answer='Yes. The available USB-chargeable/USB-powered devices include: (1) HZD mini portable desktop fan—USB rechargeable/USB powered (note: the listing says it does not come with a battery). (2) Marame 120mm 5V USB-powered fan for routers/modems/TV boxes. (3) ZARIMI compressed air duster—cordless electric with USB charging. Also, the INIU USB-C to USB-C cable is a USB-C charging cable for phones/laptops/tablets.'),
 'answer': 'Yes. The available USB-chargeable/USB-powered devices include: (1) HZD mini portable desktop fan—USB rechargeable/USB powered (note: the listing says it does not come with a battery). (2) Marame 120mm 5V USB-powered fan for routers/modems/TV boxes. (3) ZARIMI compressed air duster—cordless electric with USB charging. Also, the INIU USB-C to USB-C cable is a USB-C charging cable for phones/laptops/tablets.',
 'question': 'Any usb chargeable devices?',
 'retrieved_context_ids': ['B0C9QZS95R',
  'B0BXC72RLD',
  'B0BM9THPDQ',
 

In [16]:
print(output)

{'data_object': RAG_GenerationResponse(answer='Yes. The available USB-chargeable/USB-powered devices include: (1) HZD mini portable desktop fan—USB rechargeable/USB powered (note: the listing says it does not come with a battery). (2) Marame 120mm 5V USB-powered fan for routers/modems/TV boxes. (3) ZARIMI compressed air duster—cordless electric with USB charging. Also, the INIU USB-C to USB-C cable is a USB-C charging cable for phones/laptops/tablets.'), 'answer': 'Yes. The available USB-chargeable/USB-powered devices include: (1) HZD mini portable desktop fan—USB rechargeable/USB powered (note: the listing says it does not come with a battery). (2) Marame 120mm 5V USB-powered fan for routers/modems/TV boxes. (3) ZARIMI compressed air duster—cordless electric with USB charging. Also, the INIU USB-C to USB-C cable is a USB-C charging cable for phones/laptops/tablets.', 'question': 'Any usb chargeable devices?', 'retrieved_context_ids': ['B0C9QZS95R', 'B0BXC72RLD', 'B0BM9THPDQ', 'B0C8DBH

### RAG pipeline with grounding context

In [17]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of item used to answer the question")
    description: str = Field(description="Description of the item used to answer the question")

class RAG_GenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")


usually would have a seperate non vector db to store the item description and shortened description

In [18]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens, 
            "total_tokens": response.usage.total_tokens
        }

    return response.data[0].embedding

def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="amazon-items-collection-01",
        query=query_embedding, 
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scored = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocess_description"])
        similarity_scored.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])


    return {
        "retrieved_context_ids":  retrieved_context_ids, 
        "retrieved_context": retrieved_context,
        "similarity_scored": similarity_scored,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- if you are describing multiple products, list them out as a list

Context:
{preprocessed_context}

Question:
{question}
    """

    return prompt

def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ], 
        reasoning={"effort": "none"},
        response_model=RAG_GenerationResponse
    )

    return response

def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer, 
        "references": answer.references,
        "question": question, 
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

output = rag_pipeline("Any usb chargeable devices?")

output 

{'data_object': RAG_GenerationResponse(answer='Yes—several available products are USB chargeable / USB powered:\n\n- **HZD Desk Fan Rechargeable (B0BXC72RLD)**: USB-rechargeable mini portable fan.\n- **ZARIMI Compressed Air Duster Electric (B0C8DBH7ZT)**: Cordless electric air duster with USB charging design.\n- **Marame 120mm 5v USB Powered Fan (B0BRJS644Z)**: Powers via USB (with speed controller).\n- **Cruise Power Strip Foldable (B0BM9THPDQ)**: Includes USB-C outlets (charges devices via the power strip).', references=[RAGUsedContext(id='B0BXC72RLD', description='HZD Desk Fan Rechargeable, USB rechargeable mini portable fan (note: comes with USB cable; rechargeable fan).'), RAGUsedContext(id='B0C8DBH7ZT', description='ZARIMI Compressed air duster electric; cordless with USB charging design.'), RAGUsedContext(id='B0BRJS644Z', description='Marame 120mm 5v USB powered fan; USB-powered via cable.'), RAGUsedContext(id='B0BM9THPDQ', description='Cruise power strip with USB-C outlets (no 

In [19]:
print(output["answer"])

Yes—several available products are USB chargeable / USB powered:

- **HZD Desk Fan Rechargeable (B0BXC72RLD)**: USB-rechargeable mini portable fan.
- **ZARIMI Compressed Air Duster Electric (B0C8DBH7ZT)**: Cordless electric air duster with USB charging design.
- **Marame 120mm 5v USB Powered Fan (B0BRJS644Z)**: Powers via USB (with speed controller).
- **Cruise Power Strip Foldable (B0BM9THPDQ)**: Includes USB-C outlets (charges devices via the power strip).


now lets bump up the amount of data being retrieved

In [20]:

output = rag_pipeline("Any usb chargeable devices?", top_k=10)

In [21]:
print(output["answer"])

Yes—there are several USB-chargeable devices/products available, including:
- HZD Rechargeable Mini Portable USB Desk Fan (USB powered; 3 speed levels)
- USB Powered 120mm Fan with Speed Controller for routers/modems/TV boxes/etc. (runs on USB power)
- 1TB USB Flash Drive (USB data storage; plugs into USB ports)
- iPhone 8-pin Lightning Charger Cables (USB to Lightning for iPhones/iPads)
- USB-C to USB-C 100W PD Charging Cable (charges USB-C devices like phones/tablets/laptops)
- Cruise Power Strip with USB-C outlets (lets you charge devices via built-in USB-C ports)


lets see if anything changes with 15 items

In [22]:
output = rag_pipeline("Any usb chargeable devices?", top_k=15)

In [23]:
print(output)

{'data_object': RAG_GenerationResponse(answer='Yes—several of the available products are USB-chargeable, including:\n\n- [USB C to USB C Cable (INIU) – 6.6ft, 100W PD 5A] (charges devices that support USB-C charging)\n- [HZD Mini Rechargeable USB Desk Fan] (USB rechargeable; powered via USB)\n- [Marame Cruise Power Strip w/ USB-C outlets] (lets you charge devices via built-in USB-C ports)\n- [ZARIMI Compressed Air Duster (cordless, rechargeable)] (USB charging design)\n- [Marame 120mm 5V USB-Powered Fan with speed controller] (USB-powered)\n- [Jesebang Wireless Earbuds (Bluetooth, charging case via Type-C)] (earbuds are charged via Type-C)\n- [Wekily Bluetooth 5.3 Headphones/Earbuds] (charging case)\n- [Pamu Wireless Earbuds (ANC)] (charging case via USB-C)\n- [B0C9QZS95R etc.] [Any USB-compatible iPhone/Lightning charging cables] (USB-powered charging for compatible iPhones)\n\nIf you tell me what device you want to charge (phone/tablet model or gadget), I can narrow it down.', refere

would run the same proceduce to get any other context from the answer - even if contect differs - i.e links from web search